# Largest Rectangle in Histogram

## Problem Description
Given an array of integers `heights` representing the histogram's bar heights where the width of each bar is 1, return the area of the largest rectangle in the histogram.

## Example
```
Input: heights = [2,1,5,6,2,3]
Output: 10
Explanation: The largest rectangle has area 10 (5*2 or 6*2)

  |
  |     X X
  |     X X
  | X   X X    
  | X X X X X X
  |_____________
    2 1 5 6 2 3

Input: heights = [2,4]
Output: 4
```

## Complexity
- **Time**: O(n) - each bar visited once
- **Space**: O(n) - stack

## Why Hard?
The challenge: For each bar, expand left and right while height is >= current bar's height. Naive approach is O(n²). Monotonic stack optimizes to O(n).

## Approach 1: Monotonic Stack (Optimal - O(n))

### Key Idea:
- Maintain an **increasing monotonic stack** of indices
- For each bar, when we encounter a smaller bar:
  - Pop all taller bars from stack
  - For each popped bar, calculate max rectangle with that bar as height
  - The popped bar can expand left until the current bar and right until the next smaller bar

### Algorithm:
1. For each bar at index i:
   - If heights[i] >= heights[stack[-1]]: push i (maintain increasing stack)
   - Else: pop from stack and calculate areas
     - For popped index h: width = i - stack[-1] - 1 (distance between boundaries)
     - area = heights[h] * width

### Example:
```
heights = [2,1,5,6,2,3]
indices  = [0,1,2,3,4,5]

i=0, h=2: push 0 → stack=[0]

i=1, h=1: 1 < 2, pop 0
  - popped index = 0, height = 2
  - width = 1 - (-1) - 1 = 1 (only bar 0)
  - area = 2 * 1 = 2
  - push 1 → stack=[1]

i=2, h=5: 5 > 1, push 2 → stack=[1,2]

i=3, h=6: 6 > 5, push 3 → stack=[1,2,3]

i=4, h=2: 2 < 6, pop 3
  - popped index = 3, height = 6
  - width = 4 - 2 - 1 = 1 (only bar 3)
  - area = 6 * 1 = 6
  - 2 < 5, pop 2
  - popped index = 2, height = 5
  - width = 4 - 1 - 1 = 2 (bars 2,3)
  - area = 5 * 2 = 10 ← maximum
  - 2 >= 1, push 4 → stack=[1,4]

i=5, h=3: 3 > 2, push 5 → stack=[1,4,5]

End: pop remaining
  - pop 5: area = 3 * (5 - 4 - 1) = 0 (just bar 5)
  - pop 4: area = 2 * (6 - 1 - 1) = 8 (bars 4,5)
  - pop 1: area = 1 * 6 = 6 (all bars)

Maximum: 10
```

In [ ]:
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        """Monotonic stack approach - O(n) time"""
        stack = []  # Increasing stack of indices
        max_area = 0
        
        for i, h in enumerate(heights):
            # Pop all taller bars
            while stack and heights[stack[-1]] > h:
                h_idx = stack.pop()
                h_val = heights[h_idx]
                
                # Width: from after left_idx to before current i
                width = i if not stack else i - stack[-1] - 1
                area = h_val * width
                
                print(f"Pop index {h_idx} (height {h_val}): width={width}, area={area}")
                max_area = max(max_area, area)
            
            stack.append(i)
            print(f"i={i}, h={h}: push {i}, stack={stack}")
        
        # Pop remaining bars
        print(f"\nProcessing remaining bars in stack: {stack}")
        while stack:
            h_idx = stack.pop()
            h_val = heights[h_idx]
            
            width = len(heights) if not stack else len(heights) - stack[-1] - 1
            area = h_val * width
            
            print(f"Pop index {h_idx} (height {h_val}): width={width}, area={area}")
            max_area = max(max_area, area)
        
        return max_area

# Test
sol = Solution()
print("Test 1: [2,1,5,6,2,3]")
result = sol.largestRectangleArea([2,1,5,6,2,3])
print(f"Result: {result}\nExpected: 10\n")

print("Test 2: [2,4]")
result = sol.largestRectangleArea([2,4])
print(f"Result: {result}\nExpected: 4\n")

## Approach 2: Brute Force (O(n²))

For each bar, expand left and right to find the largest rectangle with that bar as height.

In [ ]:
class Solution2:
    def largestRectangleArea(self, heights: List[int]) -> int:
        """Brute force - O(n²) time"""
        max_area = 0
        
        for i in range(len(heights)):
            # Expand left
            left = i
            while left > 0 and heights[left - 1] >= heights[i]:
                left -= 1
            
            # Expand right
            right = i
            while right < len(heights) - 1 and heights[right + 1] >= heights[i]:
                right += 1
            
            width = right - left + 1
            area = heights[i] * width
            max_area = max(max_area, area)
        
        return max_area

sol2 = Solution2()
print("Brute Force: [2,1,5,6,2,3]")
result2 = sol2.largestRectangleArea([2,1,5,6,2,3])
print(f"Result: {result2}")

## Test Cases

In [ ]:
test_cases = [
    ([2,1,5,6,2,3], 10),
    ([2,4], 4),
    ([0,9], 9),
    ([1,1], 2),
    ([5,4,3,2,1], 5),  # Decreasing
    ([1,2,3,4,5], 9),  # Increasing
    ([4,2,0,3,2,4,3,4], 10),
]

# Clean version without prints
class SolutionClean:
    def largestRectangleArea(self, heights: List[int]) -> int:
        stack = []
        max_area = 0
        
        for i, h in enumerate(heights):
            while stack and heights[stack[-1]] > h:
                h_idx = stack.pop()
                h_val = heights[h_idx]
                width = i if not stack else i - stack[-1] - 1
                area = h_val * width
                max_area = max(max_area, area)
            stack.append(i)
        
        while stack:
            h_idx = stack.pop()
            h_val = heights[h_idx]
            width = len(heights) if not stack else len(heights) - stack[-1] - 1
            area = h_val * width
            max_area = max(max_area, area)
        
        return max_area

print("Testing Largest Rectangle in Histogram (Monotonic Stack):")
sol_clean = SolutionClean()
for heights, expected in test_cases:
    result = sol_clean.largestRectangleArea(heights)
    status = "✓" if result == expected else "✗"
    print(f"{status} {heights} → {result} (expected {expected})")

## Key Insights

1. **Monotonic Stack Pattern**: For problems involving "next/previous" element or expanding ranges
2. **Width Calculation**: Distance between boundaries is `i - left_boundary - 1`
3. **Left boundary**: The last index with smaller height (before stack position)
4. **Right boundary**: Current index where we encounter smaller height
5. **Performance**: O(n²) brute force vs O(n) stack is game-changing for large inputs

## Why Uber asks this
- Tests advanced data structure knowledge (beyond simple arrays)
- Requires optimization thinking (O(n²) → O(n))
- Real-world: Similar pattern in resource allocation, buffer management